In [1]:
import pandas as pd

LOADING CSV DATA INTO PANDA DATAFRAME

In [7]:
df = pd.read_csv("smart_meter_data.csv")

df.head()


,meter_id,timestamp,kwh_usage,voltage,region
0,MTR466611,7/27/2024 14:32,6.572,138.7,Central
1,MTR764322,7/10/2025 4:23,4.459,149.3,North
2,MTR900801,11/20/2024 8:42,8.659,161.8,East
3,MTR403291,6/13/2025 9:48,0.366,229.4,Central
4,MTR225586,10/5/2024 9:53,6.507,184.4,South


In [12]:
df['timestamp'].dtypes 

<StringDtype(na_value=nan)>

CONVERTING COLUMNS TO UPPER CASE TO FIT INTO SNOWFLAKE DATABASE

In [16]:
df.columns = [c.upper() for c in df.columns]

df.head()



,METER_ID,TIMESTAMP,KWH_USAGE,VOLTAGE,REGION
0,MTR466611,7/27/2024 14:32,6.572,138.7,Central
1,MTR764322,7/10/2025 4:23,4.459,149.3,North
2,MTR900801,11/20/2024 8:42,8.659,161.8,East
3,MTR403291,6/13/2025 9:48,0.366,229.4,Central
4,MTR225586,10/5/2024 9:53,6.507,184.4,South


CREATING CONNECTION INTO DATAFRAME

In [13]:
import snowflake.connector
from dotenv import load_dotenv
import os

load_dotenv()

conn = None

try:
    conn = snowflake.connector.connect(
        account = os.getenv("SNOWFLAKE_ACCOUNT"),
        user = os.getenv("SNOWFLAKE_USER"),
        role = os.getenv("SNOWFLAKE_ROLE"),
        warehouse = os.getenv("SNOWFLAKE_WAREHOUSE"),
        database = os.getenv("SNOWFLAKE_DATABASE"),
        schema = os.getenv("SNOWFLAKE_SCHEMA"),
        password = os.getenv("SNOWFLAKE_PASSWORD")
    )
    print("Successfully connected to Snowflake!")
except Exception as e:
    print(f"Error: {e}")



Successfully connected to Snowflake!


TESTING VERSION

In [14]:
try:
    cur = conn.cursor() # type: ignore
    cur.execute("SELECT 1")
    print("Connection OK")
except Exception as e:
    print("Connection failed:", e)

Connection OK


LOADING TO SNOWFLAKE

In [17]:
from snowflake.connector.pandas_tools import write_pandas
import traceback


if conn:
    try:
        success, nchunks, nrows, _ = write_pandas(
        conn,
        df,                      # your pandas DataFrame
        table_name="METER_DATA"  # target table in Snowflake
        )
        print(success, nrows)
        print(f"Successfully loaded {nrows} rows into Snowflake.")
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()

True 500
Successfully loaded 500 rows into Snowflake.


CLOSE CONNECTION

In [18]:
try:
    conn.close() # type: ignore
    print("Connection closed.")
except Exception as e:
    print(f"Error closing connection: {e}")

Connection closed.
